In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

# Set plot styling
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", 35)

# logger
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

In [2]:
import abcd_helper as abcd

# load conversations data
df_convos = abcd.load_conversations()
df_turns = abcd.load_turns()
logger.info(f"{len(df_convos)} conversations and " + f"{len(df_turns)} total turns")

INFO: 10042 conversations and 220983 total turns


In [3]:
df_evals_true_slots = pd.read_parquet("../data/samples/slot_extraction_evaluation_set.parquet")

eval_convos = df_evals_true_slots['convo_id'].drop_duplicates().tolist()
logger.info(f'Sampled conversations: {len(eval_convos)}')

df_turns_samples = df_turns[df_turns['convo_id'].isin(eval_convos)]
logger.info(f'Sampled messages: {len(df_turns_samples)}')

# prepare data for semantic matcher input
from nlp.context import extract_customer_contexts
all_documents = extract_customer_contexts(df_turns_samples, min_prior_agent_utterances=1, group_by_convo=False)
logger.info(f'Customer messages to be extracted {len(all_documents)}')

INFO: Sampled conversations: 1000
INFO: Sampled messages: 22262
INFO: Customer messages to be extracted 8984


In [4]:
df_schema = abcd.load_intent_slots(as_df=True)
global_slots = set()
df_evals_true_slots['true_slots'].apply(lambda x: global_slots.update(x))

## use this if we want to take all available slots from global mandatory slots
# df_schema['mandatory_slots'].map(
#     lambda x: np.concatenate(x).tolist() if len(x) > 0 else []
# ).apply(lambda x: global_slots.update(x))

available_slots = sorted(list(global_slots))
logger.info(f"There are {len(available_slots)} available slots globally:\n{available_slots}")

INFO: There are 10 available slots globally:
['account_id', 'amount', 'email', 'name', 'order_id', 'phone', 'pin_number', 'street_address', 'username', 'zip_code']


In [5]:
from nlp.slots import SemanticMatcher, DEFAULT_SLOT_HYPOTHESES  

HYPOTHESES_BY_SLOTS = {
    "account_id": "The customer enters their account ID.",
    "amount": "The customer specifies a dollar amount or price.",
    "email": "The customer enters their email address.",
    "name": "The customer specifies a merchandise product name.",
    "order_id": "The customer specifies an order ID number.",
    "phone": "The customer provides a telephone phone number.",
    "pin_number": "The customer enters their PIN number.",
    "street_address": "The customer specifies a street address.",
    "username": "The customer writes their account username in the message.",
    "zip_code": "The customer provides a postal zip code.",
}

HYPOTHESIS_GATE = "The customer provides an identifier, contact info, code, name, or numeric amount."

THRESHOLD_BY_SLOTS = {
    "email": 0.98,
    "phone": 0.97,
    "order_id": 0.95,
    "account_id": 0.96,
    "zip_code": 0.85,
    "amount": 0.9,
    "street_address": 0.9,
}

slot_extractor = SemanticMatcher(
    model_name="facebook/bart-large-mnli",
    hypothesis_template=HYPOTHESES_BY_SLOTS,
    threshold=THRESHOLD_BY_SLOTS,
    default_threshold=0.9,
    enable_informative_gate=True,
    gate_hypothesis=HYPOTHESIS_GATE,
    gate_threshold=0.02,
    enable_contextual_disambiguation=True,
    verification_threshold=0.05,
) 

/Users/ridho-akbar/Projects/data-science/projects/nlp-dialogue-state-tracking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO: HTTP Request: GET https://huggingface.co/api/models/facebook/bart-large-mnli "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/models/facebook/bart-large-mnli/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/facebook/bart-large-mnli/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/models/facebook/bart-large-mnli "HTTP/1.1 200 OK"
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 515/515 [00:00<00:00,

In [6]:
from itertools import batched
from tqdm import tqdm

records = []
BATCH_SIZE = 128
for batch_docs in tqdm(list(batched(all_documents, BATCH_SIZE)), desc="Batches"):
    batch_texts = [
        doc.get_full_document(strip_prior_customer_utterances=True)
        for doc in batch_docs
    ]
    batch_inferred = slot_extractor.extract_batch(
        batch_texts, available_slots, 
        batch_size=BATCH_SIZE,
        return_scores=True
    )
    for doc, extracted in zip(batch_docs, batch_inferred):
        inferred_slots = list(extracted.keys())
        confidence = list(extracted.values())
        records.append(
            {
                "convo_id": doc.convo_id,
                "turn_id": doc.turn_id,
                "inferred_slots": inferred_slots,
                "confidence_scores": confidence
            }
        )

df_results = pd.DataFrame(records)
logger.info(f"Finish extracting slots from {len(df_results)} messages")

Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 71/71 [03:53<00:00,  3.29s/it]
INFO: Finish extracting slots from 8984 messages


In [7]:
from data_science.evaluation import evaluate_multilabel

df_evals = (
    df_evals_true_slots.merge(
        df_results,
        how='left', on=['convo_id', 'turn_id']
    )
)

eval_result= evaluate_multilabel(
    df_evals, true_col="true_slots", 
    pred_col="inferred_slots", 
    drop_zero_support=False
)

eval_result

Global Metric,Score
exact_match_ratio,89.3%
hamming_loss,0.0127
jaccard_similarity_samples,11.0%
micro_precision,65.4%
micro_recall,77.6%
micro_f1,71.0%
macro_precision,64.4%
macro_recall,75.7%
macro_f1,67.3%
weighted_precision,69.0%


# ---- Evals notebook ends here ---- 

## Debugging

In [8]:
# BATCH_SIZE = 64
slot_extractor_trial = SemanticMatcher(
    model_name="facebook/bart-large-mnli", 
    hypothesis_template = "The customer specifies their {0}",
    threshold=0.7,
    batch_size=BATCH_SIZE
)

doc = all_documents[2]
test_text = doc.get_full_document(strip_prior_customer_utterances=True)
# test_text = doc.customer_text
# test_text = "my username is andrew123 and I live in London"

true_slots = df_turns[
    (df_turns['convo_id']==doc.convo_id)
    & (df_turns['turn_id']==doc.turn_id)
]['slot_tags'].values.tolist()

extracted = slot_extractor_trial.extract_slots(test_text, available_slots, return_scores=True)

print("Text:\n----------")
print(test_text, "\n----------")
print("Inferred slots:", extracted)
print("True slots:", true_slots)

TypeError: SemanticMatcher.__init__() got an unexpected keyword argument 'batch_size'

In [ ]:
list(extracted.keys())

In [ ]:
pairs: list[tuple[str, str]] = []
pair_mapping: list[tuple[int, str]] = []  # (text_index, slot_name)
hypothesis_template = "The customer specifies their {0}"

for text_idx, text in enumerate([test_text]):
    if not text or not str(text).strip():
        continue
    clean_text = str(text)
    for slot in available_slots:
        clean_slot = slot.replace("_", " ")
        hyp = hypothesis_template.format(clean_slot)
        pairs.append((clean_text, hyp))
        pair_mapping.append((text_idx, slot))

In [ ]:
pairs

In [ ]:
batch_texts[0]